In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os

# === Step 0: Define Top 10 Stocks ===
# Example: top_10_symbols holds the symbols of large companies, so we can loop over them.
top_10_symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'BRK-B', 'UNH', 'JPM']


# === Step 1–9: Loop over all stocks ===
# start_date and end_date define the time range for downloading the stock data.
# time_steps is how many past data points we will feed the model each time.
start_date = '2000-01-01'
end_date = datetime.today().strftime('%Y-%m-%d')
time_steps = 60

# create_sequences function:
# Turns a 1D array of prices into overlapping sequences of length "time_steps"
# Example: if time_steps=60, X[i] contains 60 consecutive price values, y[i] the next price.
def create_sequences(data, time_steps=60):
    X, y = [], []
    for i in range(time_steps, len(data)):
        X.append(data[i - time_steps:i])
        y.append(data[i])
    return np.array(X), np.array(y)

# print_metrics_inline function:
# Prints Mean Absolute Error, Mean Squared Error, and R² directly in one line.
def print_metrics_inline(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{name} MAE: {mae:.4f}, MSE: {mse:.4f}, R²: {r2:.4f}")

# Make sure directories exist for saving models, data, plots, and metrics.
for directory in ["models", "datasets", "test_set_predictions", "training_loss", "metrics"]:
    os.makedirs(directory, exist_ok=True)

# Loop through each symbol in top_10_symbols and run the entire pipeline.
for symbol in top_10_symbols:
    print(f"\n=== Processing {symbol} ===")

    # Download stock data from Yahoo Finance and keep only the Date and Close columns.
    df = yf.download(symbol, start=start_date, end=end_date, interval='1d').reset_index()
    df = df[['Date', 'Close']].dropna()
    df.to_csv(f"datasets/{symbol}_daily_data.csv", index=False)

    # Convert the close prices to numpy array for scaling and modeling.
    close_prices = df[['Close']].values
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(close_prices)

    # Split the data into training (80%) and test (20%).
    # We subtract time_steps from the test_data so the final sequences remain valid.
    split_index = int(len(scaled_data) * 0.8)
    train_data = scaled_data[:split_index]
    test_data = scaled_data[split_index - time_steps:]

    # Create sequences for training and testing with the specified time_steps window.
    X_train, y_train = create_sequences(train_data, time_steps)
    X_test, y_test = create_sequences(test_data, time_steps)

    # Build a simple LSTM model with dropout for regularization.
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
        Dropout(0.1),
        LSTM(32),
        Dropout(0.1),
        Dense(1)
    ])

    # Compile the model using mean squared error loss and Adam optimizer.
    model.compile(optimizer='adam', loss='mean_squared_error')

    # Callbacks:
    # EarlyStopping: Stop training when validation loss doesn't improve, restore best weights.
    # ModelCheckpoint: Save the best model weights to an .h5 file.
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    checkpoint_path = f'models/{symbol}_best_model.h5'
    checkpoint = ModelCheckpoint(checkpoint_path, save_best_only=True)

    # Train the model. We use:
    #  - validation_data to see how the model performs on unseen data each epoch
    #  - batch_size=32 controls how many samples are processed before updating weights
    #  - epochs=100 sets a maximum of 100 epochs unless stopped early
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=100,
        batch_size=32,
        callbacks=[early_stop, checkpoint],
        verbose=0
    )

    # Generate predictions for training and test sets.
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)

    # Convert scaled predictions back to original prices using inverse_transform.
    train_preds_inv = scaler.inverse_transform(train_preds)
    test_preds_inv = scaler.inverse_transform(test_preds)
    y_train_inv = scaler.inverse_transform(y_train)
    y_test_inv = scaler.inverse_transform(y_test)

    # Print training and test metrics (MAE, MSE, R²) inline.
    print_metrics_inline("Train", y_train_inv, train_preds_inv)
    print_metrics_inline("Test", y_test_inv, test_preds_inv)

    # === Plot Test Set Predictions ===
    # Example: the actual and predicted prices are plotted for visual comparison and saved as PNG.
    plt.figure(figsize=(12, 6))
    plt.plot(y_test_inv, label='Actual Price')
    plt.plot(test_preds_inv, label='Predicted Price')
    plt.title(f'{symbol} Stock Price Prediction (Test Set)')
    plt.xlabel('Time Steps')
    plt.ylabel('Price (USD)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'test_set_predictions/{symbol}_lstm_test_plot.png')
    plt.close()

    # === Plot Training Loss ===
    # Visualize how the training and validation loss change over the epochs.
    plt.figure(figsize=(8, 4))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{symbol} - Training vs Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'training_loss/{symbol}_lstm_loss_plot.png')
    plt.close()

    # === Predict Next Day Price ===
    # Uses the last 60 scaled points to predict the next day's price.
    last_60 = scaled_data[-time_steps:]
    last_60 = np.expand_dims(last_60, axis=0)
    next_day_scaled = model.predict(last_60)
    next_day_price = scaler.inverse_transform(next_day_scaled)
    print(f"Predicted next day's closing price for {symbol}: ${next_day_price[0][0]:.2f}")

    # === Metrics ===
    # Inverse-transform predictions and true values, compute metrics, and save results to CSV.
    train_preds_inv = scaler.inverse_transform(train_preds)
    test_preds_inv = scaler.inverse_transform(test_preds)
    y_train_inv = scaler.inverse_transform(y_train)
    y_test_inv = scaler.inverse_transform(y_test)

    mae_train = mean_absolute_error(y_train_inv, train_preds_inv)
    mse_train = mean_squared_error(y_train_inv, train_preds_inv)
    r2_train = r2_score(y_train_inv, train_preds_inv)
    mae_test = mean_absolute_error(y_test_inv, test_preds_inv)
    mse_test = mean_squared_error(y_test_inv, test_preds_inv)
    r2_test = r2_score(y_test_inv, test_preds_inv)

    results = {
        "symbol": symbol,
        "train_mae": mae_train,
        "train_mse": mse_train,
        "train_r2": r2_train,
        "test_mae": mae_test,
        "test_mse": mse_test,
        "test_r2": r2_test,
        "next_day_prediction": next_day_price[0][0]
    }

    # Convert results to a DataFrame and append (mode='a') to a CSV file with a header only once.
    df_results = pd.DataFrame([results])
    df_results.to_csv(f"metrics/{symbol}_lstm_model_metrics.csv", mode='a',
                      header=not os.path.exists("model_metrics.csv"), index=False)


2025-07-13 07:37:21.926861: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-13 07:37:22.160215: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-13 07:37:22.256155: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752392242.414764   92253 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752392242.491984   92253 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752392242.791877   92253 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin


=== Processing AAPL ===


/tmp/ipykernel_92253/3177158855.py:52: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=start_date, end=end_date, interval='1d').reset_index()
[*********************100%***********************]  1 of 1 completed
2025-07-13 07:37:26.911294: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
Train MAE: 0.2884, MSE: 0.3603, R²: 0.9988
Test MAE: 2.8819, MSE: 15.2061, R²: 0.9896
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Predicted next day's closing price for AAPL: $210.94

=== Processing MSFT ===


/tmp/ipykernel_92253/3177158855.py:52: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=start_date, end=end_date, interval='1d').reset_index()
[*********************100%***********************]  1 of 1 completed
/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
Train MAE: 1.9645, MSE: 8.2340, R²: 0.9971
Test MAE: 5.6399, MSE: 57.2835, R²: 0.9925
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
Predicted next day's closing price for BRK-B: $473.42

=== Processing UNH ===


/tmp/ipykernel_92253/3177158855.py:52: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=start_date, end=end_date, interval='1d').reset_index()
[*********************100%***********************]  1 of 1 completed
/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
Train MAE: 3.3705, MSE: 16.6446, R²: 0.9966
Test MAE: 7.6040, MSE: 137.8177, R²: 0.9810
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
Predicted next day's closing price for UNH: $307.27

=== Processing JPM ===


/tmp/ipykernel_92253/3177158855.py:52: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=start_date, end=end_date, interval='1d').reset_index()
[*********************100%***********************]  1 of 1 completed
/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
Train MAE: 0.8914, MSE: 2.0354, R²: 0.9967
Test MAE: 3.2732, MSE: 18.1398, R²: 0.9926
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
Predicted next day's closing price for JPM: $282.00
